# Atividade Prática: Desenvolvendo um Sistema de RH com SQLAlchemy


In [1]:
from pathlib import Path

import pandas as pd
from sqlalchemy import (
    Column,
    Float,
    ForeignKey,
    Integer,
    MetaData,
    String,
    Table,
    create_engine,
    func,
    insert,
    select,
    text,
    update,
)
from sqlalchemy.orm import Mapped, declarative_base, mapped_column, relationship, sessionmaker

DB_PATH = Path('sistema_rh.db')
DB_PATH.unlink(missing_ok=True)  # recomeça do zero a cada execução do notebook

## Nível 1: Básico — Configuração e SQL Puro com Segurança

### 1.1 Conexão com o banco SQLite


In [2]:
engine = create_engine(f'sqlite:///{DB_PATH}')
engine

Engine(sqlite:///sistema_rh.db)

### 1.2 Criando a tabela `funcionarios` com SQL puro


In [3]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS funcionarios (
            id      INTEGER PRIMARY KEY AUTOINCREMENT,
            nome    TEXT    NOT NULL,
            cargo   TEXT    NOT NULL,
            salario REAL    NOT NULL
        )
    '''))

### 1.3 Inserção segura (simulando um formulário web)

Os valores chegam do "formulário" como um dicionário e são passados separadamente do SQL,
através dos placeholders `:nome`, `:cargo` e `:salario`.


In [4]:
formulario = {'nome': 'Ana Souza', 'cargo': 'Desenvolvedor Júnior', 'salario': 4500.00}

with engine.begin() as conn:
    conn.execute(
        text('INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)'),
        formulario,
    )

Cadastrando mais alguns funcionários (o mesmo `INSERT` parametrizado aceita uma lista de dicionários):


In [5]:
outros_funcionarios = [
    {'nome': 'Bruno Lima', 'cargo': 'Desenvolvedor Júnior', 'salario': 4200.00},
    {'nome': 'Carla Mendes', 'cargo': 'Desenvolvedor Pleno', 'salario': 7800.00},
    {'nome': 'Diego Alves', 'cargo': 'Desenvolvedor Sênior', 'salario': 12500.00},
    {'nome': 'Elisa Rocha', 'cargo': 'Analista de RH', 'salario': 5200.00},
    {'nome': 'Fábio Nunes', 'cargo': 'Analista de RH', 'salario': 5600.00},
]

with engine.begin() as conn:
    conn.execute(
        text('INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)'),
        outros_funcionarios,
    )

#### Demonstração: o que um atacante tentaria

Imagine que alguém digite o texto abaixo no campo "nome" do formulário:


In [ ]:
nome_malicioso = "Hacker', 'x', 0); DROP TABLE funcionarios; --"

sql_concatenado = f"INSERT INTO funcionarios (nome, cargo, salario) VALUES ('{nome_malicioso}', 'Estagiário', 1500)"
print(sql_concatenado)

INSERT INTO funcionarios (nome, cargo, salario) VALUES ('Hacker', 'x', 0); DROP TABLE funcionarios; --', 'Estagiário', 1500)


In [7]:
with engine.begin() as conn:
    conn.execute(
        text('INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)'),
        {'nome': nome_malicioso, 'cargo': 'Estagiário', 'salario': 1500.00},
    )
    
    conn.execute(text('DELETE FROM funcionarios WHERE nome = :nome'), {'nome': nome_malicioso})

print('Tabela continua existindo e o texto malicioso foi tratado como um nome comum.')

Tabela continua existindo e o texto malicioso foi tratado como um nome comum.


#### Pergunta reflexiva

Quando a entrada do usuário é colada no texto do comando, o banco não tem como distinguir o que é
código SQL escrito pelo programador do que é dado digitado pelo usuário. Basta um apóstrofo (`'`)
para o atacante sair da string e injetar comandos próprios (ler dados de outras tabelas, burlar um
login com `' OR '1'='1`, apagar tabelas etc.). Isso é a **injeção de SQL (SQL Injection)**, uma das
vulnerabilidades mais exploradas em aplicações web.

Com `:nome`, `:cargo` etc., o comando SQL e os valores são enviados ao banco **separadamente**.
O banco primeiro compila a estrutura do comando e só depois encaixa os valores nos lugares
reservados, sempre como dados literais. Assim, não importa o conteúdo do valor, ele nunca é
interpretado como parte do SQL. De quebra, o driver também cuida de tipos e de escapes
automaticamente.


### 1.4 Validando a inserção com `pd.read_sql_query()`


In [8]:
df_funcionarios = pd.read_sql_query('SELECT * FROM funcionarios', engine)
df_funcionarios

,id,nome,cargo,salario
0,1,Ana Souza,Desenvolvedor Júnior,4500.0
1,2,Bruno Lima,Desenvolvedor Júnior,4200.0
2,3,Carla Mendes,Desenvolvedor Pleno,7800.0
3,4,Diego Alves,Desenvolvedor Sênior,12500.0
4,5,Elisa Rocha,Analista de RH,5200.0
5,6,Fábio Nunes,Analista de RH,5600.0


## Nível 2: Intermediário — SQLAlchemy Core (Automatização Programática)

### 2.1 Definindo a tabela `projetos` com `Table`, `MetaData` e `Column`


In [9]:
metadata = MetaData()

projetos = Table(
    'projetos',
    metadata,
    Column('id', Integer, primary_key=True, autoincrement=True),
    Column('nome', String(100), nullable=False),
    Column('orcamento', Float, nullable=False),
    Column('status', String(30), nullable=False),
)

metadata.create_all(engine)
list(metadata.tables)

['projetos']

### 2.2 Inserção em lote (bulk insert)


In [10]:
lista_de_projetos = [
    {'nome': 'Portal do Colaborador', 'orcamento': 85000.00, 'status': 'Em andamento'},
    {'nome': 'Folha de Pagamento 2.0', 'orcamento': 120000.00, 'status': 'Planejado'},
    {'nome': 'App de Ponto Eletrônico', 'orcamento': 64000.00, 'status': 'Em andamento'},
    {'nome': 'Dashboard de Indicadores', 'orcamento': 30000.00, 'status': 'Concluído'},
]

with engine.begin() as conn:
    resultado = conn.execute(insert(projetos), lista_de_projetos)

print(f'{resultado.rowcount} projetos inseridos.')
pd.read_sql_query(select(projetos), engine)

4 projetos inseridos.


,id,nome,orcamento,status
0,1,Portal do Colaborador,85000.0,Em andamento
1,2,Folha de Pagamento 2.0,120000.0,Planejado
2,3,App de Ponto Eletrônico,64000.0,Em andamento
3,4,Dashboard de Indicadores,30000.0,Concluído


### 2.3 Reajuste salarial para os Desenvolvedores Júnior

A tabela `funcionarios` foi criada com SQL puro, então usamos `autoload_with` para que o
SQLAlchemy leia a estrutura dela direto do banco (_reflection_) e a transforme em um objeto `Table`.


In [11]:
funcionarios = Table('funcionarios', metadata, autoload_with=engine)

PERCENTUAL_REAJUSTE = 0.10  # 10%

antes = pd.read_sql_query(
    select(funcionarios).where(funcionarios.c.cargo == 'Desenvolvedor Júnior'), engine
)

with engine.begin() as conn:
    resultado = conn.execute(
        update(funcionarios)
        .where(funcionarios.c.cargo == 'Desenvolvedor Júnior')
        .values(salario=funcionarios.c.salario * (1 + PERCENTUAL_REAJUSTE))
    )

print(f'{resultado.rowcount} funcionários reajustados.')

depois = pd.read_sql_query(
    select(funcionarios).where(funcionarios.c.cargo == 'Desenvolvedor Júnior'), engine
)
antes[['nome', 'salario']].merge(depois[['nome', 'salario']], on='nome', suffixes=('_antes', '_depois'))

2 funcionários reajustados.


,nome,salario_antes,salario_depois
0,Ana Souza,4500.0,4950.0
1,Bruno Lima,4200.0,4620.0


### 2.4 Relatório salarial: média por cargo


In [12]:
consulta_relatorio = (
    select(
        funcionarios.c.cargo,
        func.count(funcionarios.c.id).label('qtd_funcionarios'),
        func.round(func.avg(funcionarios.c.salario), 2).label('media_salarial'),
    )
    .group_by(funcionarios.c.cargo)
    .order_by(func.avg(funcionarios.c.salario).desc())
)

print(consulta_relatorio)  # SQL gerado pelo SQLAlchemy
pd.read_sql_query(consulta_relatorio, engine)

SELECT funcionarios.cargo, count(funcionarios.id) AS qtd_funcionarios, round(avg(funcionarios.salario), :round_1) AS media_salarial 
FROM funcionarios GROUP BY funcionarios.cargo ORDER BY avg(funcionarios.salario) DESC


,cargo,qtd_funcionarios,media_salarial
0,Desenvolvedor Sênior,1,12500.0
1,Desenvolvedor Pleno,1,7800.0
2,Analista de RH,2,5400.0
3,Desenvolvedor Júnior,2,4785.0


## Nível 3: Avançado — ORM (Orientação a Objetos e Relacionamentos)

### 3.1 e 3.2 Classes mapeadas e relacionamento entre elas

- `Departamento` 1 ⟶ N `FuncionarioORM`
- A `ForeignKey` fica em `funcionarios_orm.departamento_id`.
- `relationship(..., back_populates=...)` nas duas classes permite navegar nos dois sentidos:
  `departamento.funcionarios` e `funcionario.departamento`.

> Usamos a tabela `funcionarios_orm` para não conflitar com a tabela `funcionarios`
> criada em SQL puro no Nível 1 (que não possui a coluna de departamento).


In [13]:
Base = declarative_base()


class Departamento(Base):
    __tablename__ = 'departamentos'

    id: Mapped[int] = mapped_column(primary_key=True)
    nome: Mapped[str] = mapped_column(String(100), unique=True)

    funcionarios: Mapped[list['FuncionarioORM']] = relationship(back_populates='departamento')

    def __repr__(self):
        return f'Departamento(id={self.id}, nome={self.nome!r})'


class FuncionarioORM(Base):
    __tablename__ = 'funcionarios_orm'

    id: Mapped[int] = mapped_column(primary_key=True)
    nome: Mapped[str] = mapped_column(String(100))
    cargo: Mapped[str] = mapped_column(String(50))
    salario: Mapped[float]
    departamento_id: Mapped[int] = mapped_column(ForeignKey('departamentos.id'))

    departamento: Mapped['Departamento'] = relationship(back_populates='funcionarios')

    def __repr__(self):
        return f'FuncionarioORM(id={self.id}, nome={self.nome!r}, cargo={self.cargo!r})'


Base.metadata.create_all(engine)
list(Base.metadata.tables)

['departamentos', 'funcionarios_orm']

### 3.3 Sessão, criação do departamento e dos seus funcionários


In [ ]:
Session = sessionmaker(bind=engine)

with Session() as session:
    tecnologia = Departamento(nome='Tecnologia')
    tecnologia.funcionarios.append(FuncionarioORM(nome='Gabriel Costa', cargo='Desenvolvedor Júnior', salario=4300.00))
    tecnologia.funcionarios.append(FuncionarioORM(nome='Helena Dias', cargo='Desenvolvedor Pleno', salario=8100.00))

    recursos_humanos = Departamento(
        nome='Recursos Humanos',
        funcionarios=[
            FuncionarioORM(nome='Igor Martins', cargo='Analista de RH', salario=5400.00),
            FuncionarioORM(nome='Júlia Ramos', cargo='Coordenadora de RH', salario=9200.00),
        ],
    )

    session.add_all([tecnologia, recursos_humanos])
    session.commit()

#### Navegando de objeto para objeto


In [15]:
with Session() as session:
    for departamento in session.scalars(select(Departamento).order_by(Departamento.nome)):
        print(f'{departamento.nome}:')
        for funcionario in departamento.funcionarios:
            print(f'  - {funcionario.nome} ({funcionario.cargo}) R$ {funcionario.salario:,.2f}')

    helena = session.scalars(select(FuncionarioORM).where(FuncionarioORM.nome == 'Helena Dias')).one()
    print(f'\n{helena.nome} trabalha no departamento {helena.departamento.nome}.')

Recursos Humanos:
  - Igor Martins (Analista de RH) R$ 5,400.00
  - Júlia Ramos (Coordenadora de RH) R$ 9,200.00
Tecnologia:
  - Gabriel Costa (Desenvolvedor Júnior) R$ 4,300.00
  - Helena Dias (Desenvolvedor Pleno) R$ 8,100.00

Helena Dias trabalha no departamento Tecnologia.


Conferindo o resultado final com um `JOIN` montado pelo próprio ORM:


In [16]:
consulta_join = (
    select(
        Departamento.nome.label('departamento'),
        FuncionarioORM.nome.label('funcionario'),
        FuncionarioORM.cargo,
        FuncionarioORM.salario,
    )
    .join(FuncionarioORM.departamento)
    .order_by(Departamento.nome, FuncionarioORM.nome)
)

pd.read_sql_query(consulta_join, engine)

,departamento,funcionario,cargo,salario
0,Recursos Humanos,Igor Martins,Analista de RH,5400.0
1,Recursos Humanos,Júlia Ramos,Coordenadora de RH,9200.0
2,Tecnologia,Gabriel Costa,Desenvolvedor Júnior,4300.0
3,Tecnologia,Helena Dias,Desenvolvedor Pleno,8100.0
